# Notebook 02 : $\Lambda$ CDM Fitting with Official WMAP9 Likelihood

**Author:** Cherian P Ittyipe

This notebook performs a Bayesian fit of the flat $\Lambda$ CDM model to the WMAP nine-year CMB temperature data using the official WMAP9 Fortran likelihood. It covers likelihood validation, CAMB-based theory predictions, maximum-a-posteriori optimization, MCMC sampling, convergence diagnostics, and comparison with the published WMAP9 constraints.

## Parameters
| Symbol | Description | Prior |
|--------|-------------|-------|
| ombh2 | $\Omega_{b} h^2$ | [0.018, 0.028] |
| omch2 | $\Omega_{c} h^2$ | [0.08, 0.16] |
| H0 | $H_0$ km/s/Mpc | [60, 80] |
| ns | $n_s$ | [0.9, 1.05] |
| ln10As | $ln(10^{10} A_s)$ | [2.9, 3.3] |
| tau | $\tau$ | [0.04, 0.20] |

## Initialising Imports and Paths

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, ctypes, warnings
warnings.filterwarnings('ignore')
import camb
import emcee
import scipy.optimize as opt
import corner

BASE       = '/Users/cherianpi/Desktop/WMAP+DESI'
LIKE_DIR   = BASE + '/wmap_likelihood_v5'
PY_DIR     = BASE + '/Python files'
SO_PATH    = PY_DIR + '/libwmap9.so'
CHAINS_DIR = BASE + '/wmap_lcdm_wmap9_chains_v5'
WMAP_TT    = BASE + '/wmap_tt_spectrum_9yr_v5.txt'

for k, v in dict(LIKE_DIR=LIKE_DIR, SO_PATH=SO_PATH, CHAINS_DIR=CHAINS_DIR).items():
    status = 'OK' if os.path.exists(v) else 'NOT FOUND'
    print(f'{k:12s}: [{status}]  {v}')
print('Imports OK')

## Cell 1 : Compile the WMAP9 Library
The WMAP9 likelihood must be compiled once into a platform-specific shared library, `libwmap9.so`. Compilation requires `gfortran`, `cfitsio`, `LAPACK`, and `BLAS`.

In [ ]:
import subprocess

if os.path.exists(SO_PATH):
    print(f'libwmap9.so already exists .\n  {SO_PATH}')
else:
    CFITSIO = '/usr/local'
    sources = [
        'healpix_types.f90', 'read_fits.f90', 'read_archive_map.f90',
        'br_mod_dist.f90', 'WMAP_9yr_options.F90', 'WMAP_9yr_util.f90',
        'WMAP_9yr_gibbs.F90', 'WMAP_9yr_tt_pixlike.F90',
        'WMAP_9yr_tt_beam_ptsrc_chisq.f90', 'WMAP_9yr_teeebb_pixlike.F90',
        'WMAP_9yr_tetbeebbeb_pixlike.F90', 'WMAP_9yr_likelihood.F90',
    ]
    srcs = ' '.join(os.path.join(LIKE_DIR, s) for s in sources)
    cmd  = (
        f'cd {LIKE_DIR} && gfortran -O2 -fPIC -DOPTIMIZE '
        f'-I{CFITSIO}/include -shared -o {SO_PATH} '
        f'{srcs} -L{CFITSIO}/lib -lcfitsio -llapack -lblas'
    )
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print(f'Compiled OK: {SO_PATH}')
    else:
        print('FAILED\n', r.stderr[-3000:])
        raise RuntimeError('Compilation failed')

## Cell 2: Load Library &  Likelihood initialisation

After compilation, the notebook loads `libwmap9.so` through Python’s `ctypes` interface and initializes the official WMAP9 likelihood. The Fortran code expects to be run from the likelihood directory so that it can locate its data files.
On macOS ARM systems compiled with `gfortran`, Fortran module procedures are name-mangled. The notebook therefore explicitly resolves the following entry points:

- `___wmap_likelihood_9yr_MOD_wmap_likelihood_init`
- `___wmap_likelihood_9yr_MOD_wmap_likelihood_compute`

The initialization step confirms that the likelihood version and data files are available before any cosmological parameters are evaluated.

In [ ]:
lib = ctypes.CDLL(SO_PATH)
DP  = ctypes.c_double
DPA = ctypes.POINTER(DP)

# gfortran macOS module
INIT_SYM    = '__wmap_likelihood_9yr_MOD_wmap_likelihood_init'
COMPUTE_SYM = '__wmap_likelihood_9yr_MOD_wmap_likelihood_compute'

fn_init = getattr(lib, INIT_SYM)
fn_init.restype  = None
fn_init.argtypes = []

fn_compute = getattr(lib, COMPUTE_SYM)
fn_compute.restype  = None
fn_compute.argtypes = [DPA, DPA, DPA, DPA, DPA]

print(f'Symbols found:')
print(f'  init    : {INIT_SYM}')
print(f'  compute : {COMPUTE_SYM}')

_orig_dir = os.getcwd()
os.chdir(LIKE_DIR)
print(f'\ncwd -> {os.getcwd()}')
print('Calling wmap_likelihood_init ...')
fn_init()
os.chdir(_orig_dir)
print('Init complete.')

### Cell 2.1 : Likelihood Validation
Before fitting the model, the likelihood implementation is checked against the official WMAP9 test spectrum, `test_cls_v5.dat`. This verification ensures that the Python-to-Fortran interface, spectrum ordering, and likelihood normalization are correct.

In [ ]:
NUM_WMAP = 8
COMP_NAMES = [
    'TT high-l MASTER',       'TT low-l Gibbs',
    'TT low-l det',            'Beam+ptsrc',
    'TE high-l chi2',          'TE high-l det',
    'lowl TT/TE/EE/BB chi2',   'lowl TT/TE/EE/BB det',
]

def wmap_like(cltt, clte, clee, clbb):
    def a(x): return np.ascontiguousarray(x, dtype=np.float64)
    lk = np.zeros(NUM_WMAP, dtype=np.float64)
    fn_compute(
        a(cltt).ctypes.data_as(DPA), a(clte).ctypes.data_as(DPA),
        a(clee).ctypes.data_as(DPA), a(clbb).ctypes.data_as(DPA),
        lk.ctypes.data_as(DPA)
    )
    return 2.0 * lk.sum(), 2.0 * lk

# Verify against official test_cls_v5.dat 
print('Verification against official test spectrum (test_cls_v5.dat):')
tc = np.loadtxt(LIKE_DIR + '/data/test_cls_v5.dat')
# columns: l, TT, EE, BB, TE, PP
lt, lv = wmap_like(tc[:,1], tc[:,4], tc[:,2], tc[:,3])

print(f"{'Component':38s}  -2lnL")
print('-'*55)
for n, v in zip(COMP_NAMES, lv):
    print(f'{n:38s}  {v:12.4f}')
print('-'*55)
print(f"{'TOTAL -2lnL':38s}  {lt:12.4f}")
print(f"{'Expected (Gibbs mode)':38s}  7557.9660")
print(f"{'Difference':38s}  {lt - 7557.9659905:12.4f}")
print('\nDifferences of O(0.001) between platforms are normal.')

## Cell 3 : CAMB Theory Spectra

CAMB computes the theoretical CMB temperature and polarization spectra for a given set of cosmological parameters. The notebook evaluates TT, TE, EE, and BB spectra over the multipole range required by the WMAP9 likelihood.

CAMB returns $D_\ell $= $\ell(\ell+1)/(2π) C_{\ell}$
The WMAP9 Fortran likelihood expects the raw $C_{\ell}$ spectra, so the conversion is  $C_{\ell}= 2π / \ell(\ell+1) × D_{\ell}$ \

The notebook includes this conversion explicitly to avoid a common units mismatch between CAMB output and the likelihood input

In [ ]:
ELL_WMAP = np.arange(2, 1201)   # length 1199

def get_cls_camb(params):
    """
    params : dict with keys ombh2, omch2, H0, ns, ln10As, tau
    Returns: Dl_tt, Dl_te, Dl_ee, Dl_bb  each shape (1199,)
             in units muK^2  —  Dl = l(l+1)/(2pi) * Cl
             (This is what the WMAP9 Fortran likelihood expects.)
    """
    cp = camb.CAMBparams()
    cp.set_cosmology(H0=params['H0'], ombh2=params['ombh2'],
                     omch2=params['omch2'], tau=params['tau'])
    cp.InitPower.set_params(ns=params['ns'],
                             As=np.exp(params['ln10As']) * 1e-10)
    cp.set_for_lmax(1250, lens_potential_accuracy=0)
    res = camb.get_results(cp)
    Dl  = res.get_cmb_power_spectra(cp, CMB_unit='muK', raw_cl=False)['total']
    ell = ELL_WMAP
    # Pass Dl directly — no conversion needed
    # Columns: TT=0, EE=1, BB=2, TE=3
    return Dl[ell, 0], Dl[ell, 3], Dl[ell, 1], Dl[ell, 2]
# p_test = dict(ombh2=0.02264, omch2=0.1138, H0=70.0, ns=0.972, ln10As=3.089, tau=0.089)
# lc, _ = wmap_like(*get_cls_camb(p_test))
# print(f'CAMB test params  ->  -2lnL = {lc:.4f}')
# print(f'Official test cls ->  -2lnL = {lt:.4f}')
# print('(Should be close; exact match only for the official test_cls spectrum.)')

In [ ]:
cl_tt_c, cl_te_c, cl_ee_c, cl_bb_c = get_cls_camb(p_test)
lc, lv = wmap_like(cl_tt_c, cl_te_c, cl_ee_c, cl_bb_c)
print(f'CAMB test params  ->  -2lnL = {lc:.4f}')
print(f'Official test cls ->  -2lnL = {lt:.4f}')
print(f'Difference        :   {lc - lt:.4f}')

In [ ]:
import camb
import numpy as np

p_test = dict(ombh2=0.02264, omch2=0.1138, H0=70.0, ns=0.972, ln10As=3.089, tau=0.089)
cp = camb.CAMBparams()
cp.set_cosmology(H0=p_test['H0'], ombh2=p_test['ombh2'],
                 omch2=p_test['omch2'], tau=p_test['tau'])
cp.InitPower.set_params(ns=p_test['ns'], As=np.exp(p_test['ln10As'])*1e-10)
cp.set_for_lmax(1250, lens_potential_accuracy=0)
res = camb.get_results(cp)
Dl  = res.get_cmb_power_spectra(cp, CMB_unit='muK', raw_cl=False)['total']

print("CAMB 'total' array columns at ell=200:")
print(f"  col 0 : {Dl[200,0]:.2f}  (should be TT ~ 5000-6000 uK^2)")
print(f"  col 1 : {Dl[200,1]:.2f}  (should be EE ~   30-50  uK^2)")
print(f"  col 2 : {Dl[200,2]:.2f}  (should be BB ~    0     uK^2)")
print(f"  col 3 : {Dl[200,3]:.2f}  (should be TE ~ 100-200  uK^2)")

# Also check what the official test_cls_v5.dat has at ell=200
tc = np.loadtxt(LIKE_DIR + '/data/test_cls_v5.dat')
print(f"\nOfficial test_cls_v5.dat at ell=200 (row 198):")
print(f"  col 1 TT : {tc[198,1]:.6f}")
print(f"  col 2 EE : {tc[198,2]:.6f}")
print(f"  col 3 BB : {tc[198,3]:.6f}")
print(f"  col 4 TE : {tc[198,4]:.6f}")

# Cross-check: are the CAMB raw Cl magnitudes in the right ballpark?
ELL_WMAP = np.arange(2, 1201)
fac = 2.0 * np.pi / (ELL_WMAP * (ELL_WMAP + 1.0))
print(f"\nCAMB raw Cl_TT at ell=200 : {Dl[200,0]*fac[198]:.6e}  uK^2")
print(f"Official  Cl_TT at ell=200 : {tc[198,1]:.6e}  uK^2")

## Cell 4 : MAP Best-fit
Before launching the MCMC, the notebook finds an approximate maximum-a-posteriori (MAP) point by minimizing  −2lnL using `scipy.optimize.minimize` with the Nelder–Mead algorithm. This provides a sensible starting region for the ensemble sampler. 

In [ ]:
import time

PARAM_NAMES  = ['ombh2', 'omch2', 'H0', 'ns', 'ln10As', 'tau']
PARAM_LABELS = [r'$\Omega_b h^2$', r'$\Omega_c h^2$', r'$H_0$',
                r'$n_s$', r'$\ln(10^{10}A_s)$', r'$\tau$']
PRIOR_BOUNDS = np.array([
    [0.018, 0.028],
    [0.08,  0.16 ],
    [60.,   80.  ],
    [0.9,   1.05 ],
    [2.9,   3.3  ],
    [0.04,  0.20 ],
])

def neg2lnL(theta):
    if np.any(theta < PRIOR_BOUNDS[:,0]) or np.any(theta > PRIOR_BOUNDS[:,1]):
        return 1e10
    p = dict(zip(PARAM_NAMES, theta))
    try:
        val, _ = wmap_like(*get_cls_camb(p))
        return val if np.isfinite(val) else 1e10
    except Exception:
        return 1e10

theta0 = np.array([0.02264, 0.1138, 70.0, 0.972, 3.089, 0.089])

#  Time a single evaluation first 
t0 = time.time()
val0 = neg2lnL(theta0)
t_single = time.time() - t0
print(f'-2lnL at starting point : {val0:.4f}')
print(f'Single evaluation time  : {t_single:.3f} s')
print(f'Estimated total  : {t_single * 700 / 60:.1f} min')

#  MAP optimisation 
print('\nRunning MAP optimisation (Nelder-Mead) ...')
t_start = time.time()

result = opt.minimize(
    neg2lnL, theta0, method='Nelder-Mead',
    options=dict(xatol=1e-4, fatol=1e-2, maxiter=300, disp=True)
)

t_total = time.time() - t_start
theta_map   = result.x
neg2lnL_map = result.fun

print(f'\nMAP optimisation time   : {t_total:.1f} s  ({t_total/60:.2f} min)')
print(f'Function evaluations    : {result.nfev}')
print(f'Time per evaluation     : {t_total/result.nfev:.3f} s')

print('\n MAP Best Fit ')
for name, val in zip(PARAM_NAMES, theta_map):
    print(f'  {name:12s} = {val:.6f}')
print(f'\n  -2ln(L)_MAP = {neg2lnL_map:.4f}')

In [ ]:
# MAP spectrum vs WMAP9 TT bandpower data
data_tt    = np.loadtxt(WMAP_TT)
ell_data   = data_tt[:,0].astype(int)
Dl_data    = data_tt[:,1]
sigma_data = data_tt[:,2]

p_map = dict(zip(PARAM_NAMES, theta_map))
cp_m  = camb.CAMBparams()
cp_m.set_cosmology(H0=p_map['H0'], ombh2=p_map['ombh2'],
                   omch2=p_map['omch2'], tau=p_map['tau'])
cp_m.InitPower.set_params(ns=p_map['ns'], As=np.exp(p_map['ln10As'])*1e-10)
cp_m.set_for_lmax(1250, lens_potential_accuracy=0)
Dl_map_all = camb.get_results(cp_m).get_cmb_power_spectra(
    cp_m, CMB_unit='muK', raw_cl=False)['total']
Dl_map_tt  = Dl_map_all[ell_data, 0]

fig, axes = plt.subplots(2, 1, figsize=(12, 7),
                          gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
ax = axes[0]
ax.errorbar(ell_data, Dl_data, yerr=sigma_data,
            fmt='.', ms=2, lw=0.5, color='steelblue',
            ecolor='lightblue', elinewidth=0.8, label='WMAP9 TT', zorder=2)
ax.plot(ell_data, Dl_map_tt, 'r-', lw=1.5,
        label=f'MAP  (-2lnL = {neg2lnL_map:.2f})', zorder=3)
ax.set_ylabel(r'$D_\ell\ [\mu{\rm K}^2]$', fontsize=12)
ax.set_title('WMAP9 TT — ΛCDM MAP Fit (Official Likelihood)', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(2, 1000)
ax2 = axes[1]
ax2.axhline(0, color='k', lw=0.8)
ax2.scatter(ell_data, (Dl_data - Dl_map_tt)/sigma_data,
            s=1.2, color='steelblue', alpha=0.6)
ax2.set_xlabel(r'$\ell$', fontsize=12)
ax2.set_ylabel(r'$(D^{\rm obs}-D^{\rm th})/\sigma$', fontsize=10)
ax2.set_ylim(-5, 5)
plt.tight_layout()
plt.savefig(BASE + '/Images/map_bestfit_official.png', dpi=150)
plt.show()
print('MAP plot saved.')

## Cell 5 — MCMC with `emcee` + Official WMAP9 Likelihood

Log-posterior: flat priors inside bounds, so ln P(θ|d) = −(−2lnL)/2

Setup: 32 walkers, 1000 steps, 300 burn-in, DEMove, initialised at MAP.

In [ ]:
def log_prior(theta):
    if np.all((theta >= PRIOR_BOUNDS[:,0]) & (theta <= PRIOR_BOUNDS[:,1])):
        return 0.0
    return -np.inf

def log_likelihood_official(theta):
    p = dict(zip(PARAM_NAMES, theta))
    try:
        neg2, _ = wmap_like(*get_cls_camb(p))
        return -0.5 * neg2
    except Exception:
        return -np.inf

def log_posterior(theta):
    lp = log_prior(theta)
    return lp + log_likelihood_official(theta) if np.isfinite(lp) else -np.inf

NDIM, NWALKERS, NSTEPS, NBURN = 6, 32, 1000, 100

rng = np.random.default_rng(42)
p0  = theta_map + 1e-3 * theta_map * rng.standard_normal((NWALKERS, NDIM))
p0  = np.clip(p0, PRIOR_BOUNDS[:,0], PRIOR_BOUNDS[:,1])

print(f'Likelihood : Official WMAP9 ({NUM_WMAP} components)')
print(f'Walkers    : {NWALKERS}')
print(f'Steps      : {NSTEPS}  (burn-in: first {NBURN} discarded)')
print(f'Init       : tight Gaussian ball around MAP point')

In [ ]:
print('Running emcee MCMC with official WMAP9 likelihood ...')
sampler = emcee.EnsembleSampler(NWALKERS, NDIM, log_posterior,
                                  moves=emcee.moves.DEMove())
sampler.run_mcmc(p0, NSTEPS, progress=True)
print('\nMCMC complete.')
af = sampler.acceptance_fraction
print(f'Mean acceptance fraction: {af.mean():.3f}  (healthy: 0.2-0.5)')

### Cell 5.1 : Gelman-Rubin Convergence Diagnostic

In [ ]:
#Gelman-Rubin Convergence Diagnostic

def gelman_rubin(chain):
    nsteps, nwalkers = chain.shape
    walker_mean = chain.mean(axis=0)           
    walker_var  = chain.var(axis=0, ddof=1)    
    grand_mean = walker_mean.mean()
    B = nsteps * walker_mean.var(ddof=1) 
    W = walker_var.mean()
    V_hat = (1 - 1/nsteps) * W + (1/nsteps) * B
    R_hat = np.sqrt(V_hat / W)
    return R_hat



chain_full = sampler.get_chain(discard=NBURN)   

print('Gelman-Rubin Convergence Diagnostic ')
print(f'  Chain length used : {chain_full.shape[0]} steps x {chain_full.shape[1]} walkers')
print(f'  {"Parameter":12s}  {"R_hat":>8s}  {"Status"}')
print('  ' + '-'*42)

all_converged = True
for i, name in enumerate(PARAM_NAMES):
    R = gelman_rubin(chain_full[:, :, i])
    if R < 1.01:
        status = 'CONVERGED'
    elif R < 1.05:
        status = 'MARGINAL'
    else:
        status = 'NOT CONVERGED — run longer'
        all_converged = False
    print(f'  {name:12s}  {R:8.5f}  {status}')

print()
if all_converged:
    print('  All parameters converged (R_hat < 1.01).')
else:
    print('  WARNING: some parameters not converged.')
    print('  Recommendation: increase NSTEPS or NBURN and re-run.')


print()
print(' Effective Sample Size')
try:
    tau_ac = sampler.get_autocorr_time(discard=NBURN, quiet=True)
    print(f'  {"Parameter":12s}  {"tau_ac":>8s}  {"N_eff":>8s}')
    print('  ' + '-'*34)
    for name, t in zip(PARAM_NAMES, tau_ac):
        n_eff = chain_full.shape[0] * chain_full.shape[1] / t
        print(f'  {name:12s}  {t:8.1f}  {n_eff:8.0f}')
except Exception as e:
    print(f'  Autocorr estimate failed: {e}')
    print('  (Chain may still be too short for reliable autocorr estimate.)')

# R hat summary 
R_hats = [gelman_rubin(chain_full[:, :, i]) for i in range(NDIM)]

fig, ax = plt.subplots(figsize=(8, 3))
colors = ['green' if r < 1.01 else 'orange' if r < 1.05 else 'red' for r in R_hats]
bars = ax.barh(PARAM_LABELS, R_hats, color=colors, alpha=0.75, edgecolor='k', lw=0.7)
ax.axvline(1.01, color='green',  lw=1.5, ls='--', label='R_hat = 1.01 (converged)')
ax.axvline(1.05, color='orange', lw=1.5, ls='--', label='R_hat = 1.05 (marginal)')
ax.set_xlabel(r'$\hat{R}$ (Gelman-Rubin statistic)', fontsize=11)
ax.set_title('Gelman-Rubin Convergence Diagnostic — WMAP9 ΛCDM MCMC', fontsize=12)
ax.legend(fontsize=9)
ax.set_xlim(1.0, max(max(R_hats) * 1.05, 1.06))
for bar, r in zip(bars, R_hats):
    ax.text(r + 0.0005, bar.get_y() + bar.get_height()/2,
            f'{r:.5f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(BASE + '/Images/gelman_rubin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gelman-Rubin plot saved.')

In [ ]:
try:
    tau_ac = sampler.get_autocorr_time(quiet=True)
    print('Autocorrelation times:')
    for lbl, t in zip(PARAM_NAMES, tau_ac):
        print(f'  {lbl:10s}: tau = {t:.1f}   N_eff ~ {NWALKERS*(NSTEPS-NBURN)/t:.0f}')
except Exception as e:
    print(f'Autocorr estimate: {e}')

flat_samples = sampler.get_chain(discard=NBURN, thin=1, flat=True)
print(f'\nFlat samples shape: {flat_samples.shape}')

OUT = PY_DIR + '/wmap9_official_lcdm_mcmc_chains.npy'
np.save(OUT, flat_samples)
print(f'Saved: {OUT}')

In [ ]:
import json
from datetime import datetime

#Save chains 
CHAIN_DIR = PY_DIR + '/chains'
os.makedirs(CHAIN_DIR, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Flat post-burn-in samples 
flat_path = CHAIN_DIR + f'/wmap9_lcdm_flat_samples_{timestamp}.npy'
np.save(flat_path, flat_samples)

# 2. Full raw chain 
full_chain_path = CHAIN_DIR + f'/wmap9_lcdm_full_chain_{timestamp}.npy'
np.save(full_chain_path, sampler.get_chain())   # shape (NSTEPS, NWALKERS, NDIM)

# 3. Log-probability chain 
logprob_path = CHAIN_DIR + f'/wmap9_lcdm_logprob_{timestamp}.npy'
np.save(logprob_path, sampler.get_log_prob())   # shape (NSTEPS, NWALKERS)

# 4. Metadata JSON 
meta = {
    'timestamp'         : timestamp,
    'likelihood'        : 'Official WMAP9 (wmap_likelihood_compute, 8 components)',
    'theory_code'       : 'CAMB',
    'param_names'       : PARAM_NAMES,
    'prior_bounds'      : PRIOR_BOUNDS.tolist(),
    'theta_map'         : theta_map.tolist(),
    'neg2lnL_map'       : float(neg2lnL_map),
    'nwalkers'          : NWALKERS,
    'nsteps'            : NSTEPS,
    'nburn'             : NBURN,
    'ndim'              : NDIM,
    'flat_samples_shape': list(flat_samples.shape),
    'acceptance_fraction_mean' : float(sampler.acceptance_fraction.mean()),
    'acceptance_fraction_per_walker' : sampler.acceptance_fraction.tolist(),
    'flat_samples_file' : flat_path,
    'full_chain_file'   : full_chain_path,
    'logprob_file'      : logprob_path,
    'gelman_rubin'      : {name: float(gelman_rubin(sampler.get_chain(discard=NBURN)[:,:,i]))
                           for i, name in enumerate(PARAM_NAMES)},
    'posterior_summary' : {
        name: {
            'median' : float(np.percentile(flat_samples[:,i], 50)),
            'lo_1s'  : float(np.percentile(flat_samples[:,i], 50) - np.percentile(flat_samples[:,i], 16)),
            'hi_1s'  : float(np.percentile(flat_samples[:,i], 84) - np.percentile(flat_samples[:,i], 50)),
        }
        for i, name in enumerate(PARAM_NAMES)
    }
}

meta_path = CHAIN_DIR + f'/wmap9_lcdm_meta_{timestamp}.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print('Chains saved ')
print(f'  Directory    : {CHAIN_DIR}')
print(f'  Flat samples : {os.path.basename(flat_path)}   {flat_samples.nbytes/1e6:.1f} MB')
print(f'  Full chain   : {os.path.basename(full_chain_path)}   {sampler.get_chain().nbytes/1e6:.1f} MB')
print(f'  Log-prob     : {os.path.basename(logprob_path)}')
print(f'  Metadata     : {os.path.basename(meta_path)}')


# Load chains 
def load_chains(timestamp=None, chain_dir=CHAIN_DIR):
    """
    Load a saved chain run by timestamp.
    If timestamp is None, loads the most recent run.

    Returns
    -------
    flat_samples : array (N, NDIM)
    full_chain   : array (NSTEPS, NWALKERS, NDIM)
    logprob      : array (NSTEPS, NWALKERS)
    meta         : dict
    """
    import glob

    if timestamp is None:
        # Find most recent metadata file
        meta_files = sorted(glob.glob(chain_dir + '/wmap9_lcdm_meta_*.json'))
        if not meta_files:
            raise FileNotFoundError(f'No chain metadata found in {chain_dir}')
        meta_path = meta_files[-1]
    else:
        meta_path = chain_dir + f'/wmap9_lcdm_meta_{timestamp}.json'

    with open(meta_path) as f:
        meta = json.load(f)

    flat_samples = np.load(meta['flat_samples_file'])
    full_chain   = np.load(meta['full_chain_file'])
    logprob      = np.load(meta['logprob_file'])

    print(f'Loaded chain from: {meta_path}')
    print(f'  Flat samples : {flat_samples.shape}')
    print(f'  Full chain   : {full_chain.shape}')
    print(f'  Walkers      : {meta["nwalkers"]}')
    print(f'  Steps        : {meta["nsteps"]}  (burn-in: {meta["nburn"]})')
    print(f'  Acceptance   : {meta["acceptance_fraction_mean"]:.3f}')

    return flat_samples, full_chain, logprob, meta


## Cell 6 : Trace Plots

In [ ]:
chain = sampler.get_chain()
fig, axes = plt.subplots(NDIM, 1, figsize=(12, 10), sharex=True)
for i, (ax, lbl) in enumerate(zip(axes, PARAM_LABELS)):
    ax.plot(chain[:,:,i], alpha=0.3, lw=0.5, color='steelblue')
    ax.axvline(NBURN, color='red', lw=1.2, ls='--',
               label='burn-in cutoff' if i==0 else None)
    ax.set_ylabel(lbl, fontsize=10)
    ax.yaxis.set_label_coords(-0.08, 0.5)
axes[0].legend(fontsize=9)
axes[-1].set_xlabel('Step', fontsize=11)
fig.suptitle('MCMC Trace — WMAP9 ΛCDM (Official Likelihood)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(BASE + '/Images/mcmc_trace_official.png', dpi=150, bbox_inches='tight')
plt.show()
print('Trace plot saved.')

## Cell7 : Posterior Summary vs Published WMAP9 Values

In [ ]:
wmap9_pub = {
    'ombh2' : (0.02264, 0.00050),
    'omch2' : (0.1138,  0.0045),
    'H0'    : (70.0,    2.2),
    'ns'    : (0.972,   0.013),
    'ln10As': (3.089,   0.025),
    'tau'   : (0.089,   0.014),} # Hinshaw et al. 2013 

h_ch   = flat_samples[:,2] / 100.0
omm_ch = (flat_samples[:,0] + flat_samples[:,1]) / h_ch**2
all_s  = np.column_stack([flat_samples, omm_ch])
all_n  = PARAM_NAMES + ['Omega_m']

print(f"  {'Param':12s}  {'MAP':>10s}  {'Median':>10s}  {'-1s':>7s}  {'+1s':>7s}  {'Published':>10s}")
print('  ' + '-'*64)
for i, lbl in enumerate(all_n):
    med   = np.percentile(all_s[:,i], 50)
    lo    = med - np.percentile(all_s[:,i], 16)
    hi    = np.percentile(all_s[:,i], 84) - med
    pub   = wmap9_pub.get(lbl, (np.nan,))[0]
    map_v = theta_map[i] if i < 6 else np.nan
    print(f'  {lbl:12s}  {map_v:10.5f}  {med:10.5f}  {lo:7.5f}  {hi:7.5f}  {pub:10.5f}')
print(f'\n  MAP -2lnL = {neg2lnL_map:.4f}')

## Cell 8 : Corner Plot

In [ ]:
fig = corner.corner(
    flat_samples, labels=PARAM_LABELS,
    quantiles=[0.16, 0.50, 0.84], show_titles=True,
    title_fmt='.4f', title_kwargs={'fontsize': 9},
    truths=theta_map, truth_color='red',
    color='steelblue', smooth=1.0, bins=40,
    fill_contours=True, levels=[0.68, 0.95],
    contourf_kwargs={'alpha': 0.3}
)
fig.suptitle('WMAP9 ΛCDM — Official Likelihood Posterior', fontsize=13, y=1.01)
plt.savefig(BASE + '/Images/mcmc_corner_official.png', dpi=150, bbox_inches='tight')
plt.show()
print('Corner plot saved.')

## Cell 9: MCMC vs Official WMAP9 Published Chains

In [ ]:
def load_chain(name):
    return np.loadtxt(os.path.join(CHAINS_DIR, name))[:,1]

off_chains  = [
    load_chain('omegabh2'), load_chain('omegach2'),
    load_chain('H0'), load_chain('ns002'), load_chain('tau')
]
off_weight  = load_chain('weight')
plot_labels = [r'$\Omega_b h^2$', r'$\Omega_c h^2$', r'$H_0$', r'$n_s$', r'$\tau$']
our_idxs    = [0, 1, 2, 3, 5]

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
for ax, lbl, ochn, oidx in zip(axes, plot_labels, off_chains, our_idxs):
    ax.hist(flat_samples[:,oidx], bins=40, density=True,
            color='steelblue', alpha=0.55, label='Our MCMC\n(official like.)')
    ax.hist(ochn, bins=40, density=True, weights=off_weight,
            histtype='step', color='darkorange', lw=2.0, label='WMAP9 official\nchains')
    ax.set_xlabel(lbl, fontsize=11)
    ax.set_ylabel('P (norm.)' if ax is axes[0] else '', fontsize=10)
    ax.legend(fontsize=7)
fig.suptitle('Our MCMC (Official Likelihood) vs Published WMAP9 Chains', fontsize=12)
plt.tight_layout()
plt.savefig(BASE + '/Images/mcmc_vs_official_chains.png', dpi=150, bbox_inches='tight')
plt.show()
print('Comparison plot saved.')

## Cell 10 : Best-fit Spectrum & Posterior Envelope 
The notebook computes the TT spectrum at the posterior median and compares it with the WMAP9 data. It also constructs a 68% posterior envelope by evaluating CAMB spectra for 150 randomly selected posterior samples.

In [ ]:
theta_mcmc = np.percentile(flat_samples, 50, axis=0)
p_mc = dict(zip(PARAM_NAMES, theta_mcmc))
cp_mc = camb.CAMBparams()
cp_mc.set_cosmology(H0=p_mc['H0'], ombh2=p_mc['ombh2'],
                    omch2=p_mc['omch2'], tau=p_mc['tau'])
cp_mc.InitPower.set_params(ns=p_mc['ns'], As=np.exp(p_mc['ln10As'])*1e-10)
cp_mc.set_for_lmax(1250, lens_potential_accuracy=0)
Dl_mc_tt = camb.get_results(cp_mc).get_cmb_power_spectra(
    cp_mc, CMB_unit='muK', raw_cl=False)['total'][ell_data, 0]

# 68% envelope from 150 random posterior samples
rng_e = np.random.default_rng(99)
idx_e = rng_e.choice(len(flat_samples), size=150, replace=False)
Dl_env = []
for i in idx_e:
    p_s = dict(zip(PARAM_NAMES, flat_samples[i]))
    cp_s = camb.CAMBparams()
    cp_s.set_cosmology(H0=p_s['H0'], ombh2=p_s['ombh2'],
                       omch2=p_s['omch2'], tau=p_s['tau'])
    cp_s.InitPower.set_params(ns=p_s['ns'], As=np.exp(p_s['ln10As'])*1e-10)
    cp_s.set_for_lmax(1250, lens_potential_accuracy=0)
    Dl_env.append(camb.get_results(cp_s).get_cmb_power_spectra(
        cp_s, CMB_unit='muK', raw_cl=False)['total'][ell_data, 0])
Dl_env = np.array(Dl_env)
env_lo = np.percentile(Dl_env, 16, axis=0)
env_hi = np.percentile(Dl_env, 84, axis=0)

fig, axes = plt.subplots(2, 1, figsize=(12, 7),
                          gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
ax = axes[0]
ax.errorbar(ell_data, Dl_data, yerr=sigma_data,
            fmt='.', ms=2, lw=0.4, color='steelblue',
            ecolor='lightblue', elinewidth=0.7, label='WMAP9 TT', zorder=2)
ax.fill_between(ell_data, env_lo, env_hi,
                color='salmon', alpha=0.45, label='68% posterior envelope')
ax.plot(ell_data, Dl_mc_tt, 'r-', lw=1.5, label='MCMC median', zorder=4)
ax.plot(ell_data, Dl_map_tt, 'k--', lw=1.0, alpha=0.8, label='MAP', zorder=3)
ax.set_ylabel(r'$D_\ell\ [\mu{\rm K}^2]$', fontsize=12)
ax.set_title('WMAP9 TT — ΛCDM (Official Likelihood)', fontsize=13)
ax.legend(fontsize=9)
ax.set_xlim(2, 1000)
ax2 = axes[1]
ax2.axhline(0, color='k', lw=0.8)
ax2.fill_between(ell_data,
                 (Dl_data-env_hi)/sigma_data, (Dl_data-env_lo)/sigma_data,
                 color='salmon', alpha=0.4)
ax2.scatter(ell_data, (Dl_data-Dl_mc_tt)/sigma_data,
            s=1.2, color='red', alpha=0.7, zorder=3)
ax2.set_xlabel(r'$\ell$', fontsize=12)
ax2.set_ylabel(r'$(D^{\rm obs}-D^{\rm th})/\sigma$', fontsize=9)
ax2.set_ylim(-5, 5)
plt.tight_layout()
plt.savefig(BASE + '/Images/mcmc_official_spectrum.png', dpi=150)
plt.show()
print('Spectrum plot saved.')

This visualization shows both the best-fit model and the range of spectra allowed by the posterior. The residual panel displays the normalized difference between the observed and theoretical TT spectra.

## Cell 11 : Final Summary

The final summary reports the likelihood configuration, MCMC settings, number of retained samples, MAP likelihood value, and posterior constraints. It provides a compact record of the WMAP9-only analysis before the DESI BAO data are added in the joint fit.



In [ ]:
print('='*70)
print('  WMAP9 ΛCDM — FINAL SUMMARY (Official Likelihood)')
print('='*70)
print(f'  Likelihood   : Official WMAP9 ({NUM_WMAP} components)')
print(f'  MCMC         : {NWALKERS} walkers x {NSTEPS} steps, burn-in {NBURN}')
print(f'  Flat samples : {flat_samples.shape[0]}')
print(f'  MAP -2lnL    : {neg2lnL_map:.4f}')
print()
print(f"  {'Param':12s}  {'MAP':>10s}  {'Median':>10s}  {'-1s':>8s}  {'+1s':>8s}  {'Published':>10s}")
print('  ' + '-'*66)
for i, lbl in enumerate(PARAM_NAMES):
    med = np.percentile(flat_samples[:,i], 50)
    lo  = med - np.percentile(flat_samples[:,i], 16)
    hi  = np.percentile(flat_samples[:,i], 84) - med
    pub = wmap9_pub.get(lbl, (np.nan,))[0]
    print(f'  {lbl:12s}  {theta_map[i]:10.5f}  {med:10.5f}  {lo:8.5f}  {hi:8.5f}  {pub:10.5f}')
print('='*70)

The key result is that the independently implemented WMAP9 likelihood and MCMC pipeline recover posterior constraints consistent with the official published WMAP9 analysis, providing a solid baseline for the subsequent WMAP9 + DESI DR1 joint fit.